In [9]:

# up Flower Classification with ResNet18 and Enhanced Visualizations

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torchvision

# ==== CONFIGURATION ====
data_dir = r"D:\ml_lern\dataset"
batch_size = 16
image_size = 128
epochs = 3
learning_rate = 0.0001
model_save_path = "models/flower_resnet18.pth"

# ==== DEVICE SETUP ====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==== DATA TRANSFORMS ====
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(image_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ==== LOAD DATASET AND SPLIT ====
dataset = datasets.ImageFolder(root=data_dir, transform=transform_train)
total_size = len(dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])
val_set.dataset.transform = transform_val
test_set.dataset.transform = transform_val

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

class_names = dataset.classes
num_classes = len(class_names)
print(f"Classes found: {class_names}")
print(f"Total images: {total_size}")

# ==== MODEL SETUP ====
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# ==== LOSS AND OPTIMIZER ====
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# ==== TRAINING LOOP ====
train_losses = []
train_accuracies = []
val_accuracies = []
epoch_times = []

print("Starting training...\n")
for epoch in range(epochs):
    model.train()
    start_time = time.time()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.to(device), val_labels.to(device)
            val_outputs = model(val_images)
            _, val_preds = torch.max(val_outputs, 1)
            val_correct += (val_preds == val_labels).sum().item()
            val_total += val_labels.size(0)

    val_acc = 100 * val_correct / val_total

    end_time = time.time()
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)
    val_accuracies.append(val_acc)
    epoch_times.append(end_time - start_time)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.2f}% - Val Acc: {val_acc:.2f}%")

print("\nTraining complete!")

# ==== SAVE MODEL ====
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")


Using device: cpu
Classes found: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
Total images: 4317
Starting training...

Epoch [1/3] - Loss: 0.5681 - Acc: 79.54% - Val Acc: 86.86%
Epoch [2/3] - Loss: 0.1627 - Acc: 94.47% - Val Acc: 89.34%
Epoch [3/3] - Loss: 0.0822 - Acc: 97.58% - Val Acc: 88.41%

Training complete!
Model saved to models/flower_resnet18.pth


In [6]:
import torch
import torchvision
import torchaudio

print("PyTorch version:", torch.__version__)


PyTorch version: 2.7.1+cpu


In [5]:
import sys
print(sys.executable)


d:\ml_lern\.venv\Scripts\python.exe
